In [3]:
# import necessary packages
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import statsmodels.api as sm

from sklearn.linear_model import LinearRegression # Linear Regression Model
from sklearn.preprocessing import StandardScaler # Z-score variables
from sklearn.preprocessing import MinMaxScaler # Min-Max Normalization

from sklearn.model_selection import train_test_split # simple TT split cv

from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

import os
import numpy as np
from scipy.interpolate import BSpline, make_interp_spline

In [22]:
cb_file = 'C:\\Users\\katie\\Documents\\cpsc\\VRelax\\gestureInterface\\Codebook\\Codebook_Pan_Right.csv'
data_folder = 'C:\\Users\\katie\\Documents\\cpsc\\VRelax\\gestureInterface\\CleanedData\\'

df = pd.read_csv(cb_file, header=None)
file_name = os.path.basename(cb_file)
gesture = file_name.replace("Codebook", "")
gesture = gesture.replace("_", "")
gesture = gesture.replace(".csv", "")
row_length = df.shape[0]

num_hands_uni = 0
num_hands_bi = 0
num_direction_same = 0
num_direction_diff = 0
num_symm_s = 0
num_symm_a = 0
num_shape_l_sc = 0
num_shape_l_ss = 0
num_shape_l_sq = 0
num_shape_l_sd = 0
num_shape_l_so = 0
num_shape_r_sc = 0
num_shape_r_ss = 0
num_shape_r_sq = 0
num_shape_r_sd = 0
num_shape_r_so = 0


PanRight
['Codebook_Pan_Down.csv', 'Codebook_Pan_Left.csv', 'Codebook_Pan_Right.csv', 'Codebook_Pan_Up.csv', 'Codebook_Rot_Back_X.csv', 'Codebook_Rot_Clockwise_Y.csv', 'Codebook_Rot_Counter_Y.csv', 'Codebook_Rot_Forward_X.csv', 'Codebook_Rot_Left_Z.csv', 'Codebook_Rot_Right_Z.csv', 'Codebook_Zoom_In.csv', 'Codebook_Zoom_Out.csv']


Hands

In [12]:
curr_sub_df = pd.DataFrame()
for i in range(row_length):
    # Update curr_sub_df
    if i % 8 == 0:
        identifier = df.iloc[i,0]
        sub_num = "Sub" + identifier.split(" ")[-1] 
        sub_folder_path = data_folder + sub_num + "\\Freeform_" + sub_num + "_Sess1"
        files = os.listdir(sub_folder_path)
        #Iterates through the subject's folder and stops when it finds the file with the right gesture
        for j in range(len(files)):
            if files[j].split("_")[3] == gesture:
                curr_sub_df = pd.read_csv(os.path.join(sub_folder_path, files[j]))
                break

    # Removes rows that contain the subject number, categories, and ids
    # Trials only
    if (i % 8 != 0) and (i % 8 != 1) and (i % 8 != 2):
        temp_df = curr_sub_df.loc[(curr_sub_df['gesture_counter_UI'] == ((i%8)-2)) & (((curr_sub_df["trigger_pull_amount_right"] > 0.0) | (curr_sub_df["trigger_pull_amount_left"] > 0.0)))]
        if (len(temp_df) < 5):
            continue
        if pd.isna(df.iloc[i,11]):
            num_hands_bi += 1
        else:
            num_hands_uni += 1

61
64


Direction

In [13]:
curr_sub_df = pd.DataFrame()
for i in range(row_length):
    # Update curr_sub_df
    if i % 8 == 0:
        identifier = df.iloc[i,0]
        sub_num = "Sub" + identifier.split(" ")[-1] 
        sub_folder_path = data_folder + sub_num + "\\Freeform_" + sub_num + "_Sess1"
        files = os.listdir(sub_folder_path)
        #Iterates through the subject's folder and stops when it finds the file with the right gesture
        for j in range(len(files)):
            if files[j].split("_")[3] == gesture:
                curr_sub_df = pd.read_csv(os.path.join(sub_folder_path, files[j]))
                break

    # Removes rows that contain the subject number, categories, and ids
    # Trials only
    if (i % 8 != 0) and (i % 8 != 1) and (i % 8 != 2):
        temp_df = curr_sub_df.loc[(curr_sub_df['gesture_counter_UI'] == ((i%8)-2)) & (((curr_sub_df["trigger_pull_amount_right"] > 0.0) | (curr_sub_df["trigger_pull_amount_left"] > 0.0)))]
        if (len(temp_df) < 5):
            continue
        if pd.isna(df.iloc[i,13]):
            num_direction_diff += 1
        else:
            num_direction_same += 1

33
92


Symmetry

In [14]:
curr_sub_df = pd.DataFrame()
for i in range(row_length):
    # Update curr_sub_df
    if i % 8 == 0:
        identifier = df.iloc[i,0]
        sub_num = "Sub" + identifier.split(" ")[-1] 
        sub_folder_path = data_folder + sub_num + "\\Freeform_" + sub_num + "_Sess1"
        files = os.listdir(sub_folder_path)
        #Iterates through the subject's folder and stops when it finds the file with the right gesture
        for j in range(len(files)):
            if files[j].split("_")[3] == gesture:
                curr_sub_df = pd.read_csv(os.path.join(sub_folder_path, files[j]))
                break

    # Removes rows that contain the subject number, categories, and ids
    # Trials only
    if (i % 8 != 0) and (i % 8 != 1) and (i % 8 != 2):
        temp_df = curr_sub_df.loc[(curr_sub_df['gesture_counter_UI'] == ((i%8)-2)) & (((curr_sub_df["trigger_pull_amount_right"] > 0.0) | (curr_sub_df["trigger_pull_amount_left"] > 0.0)))]
        if (len(temp_df) < 5):
            continue
        if pd.isna(df.iloc[i,15]):
            num_symm_a += 1
        else:
            num_symm_s += 1

13
112


In [16]:
def get_gesture_name(file_name):
    gesture = file_name.replace("Codebook", "")
    gesture = gesture.replace("_", "")
    gesture = gesture.replace(".csv", "")

    return gesture

def get_percentage(count, total):
    percentage = round((count/total) * 100, 2)
    return str(percentage) + '%'

def export_csv(df, folder_path):
    export_file_path = folder_path + '\\Codebook_Analysis.csv'
    if not os.path.exists(export_file_path):
        df.to_csv(export_file_path, header=True, index=False)



def initialize_dict(df):
    df['gesture'] = np.nan
    df['shape l (SC)'] = np.nan
    df['shape l (SS)'] = np.nan
    df['shape l (SQ)'] = np.nan
    df['shape l (SD)'] = np.nan
    df['shape l (SO)'] = np.nan
    df['shape r (SC)'] = np.nan
    df['shape r (SS)'] = np.nan
    df['shape r (SQ)'] = np.nan
    df['shape r (SD)'] = np.nan
    df['shape r (SO)'] = np.nan
    df['hands (HU)'] = np.nan
    df['hands (HB)'] = np.nan
    df['direction (DS)'] = np.nan
    df['direction (DD)'] = np.nan
    df['symmetry (S)'] = np.nan
    df['symmetry (A)'] = np.nan

def add_gesture_data(df,
                     gesture, 
                     shape_l_SC, shape_l_SS, shape_l_SQ, shape_l_SD, shape_l_SO, 
                     shape_r_SC, shape_r_SS, shape_r_SQ, shape_r_SD, shape__SO,
                     hands_HU, hands_HB,
                     direction_DS, direction_DD,
                     symmetry_S, symmetry_A):
    
    df.loc[len(df.index)] = [gesture, 
                     shape_l_SC, shape_l_SS, shape_l_SQ, shape_l_SD, shape_l_SO, 
                     shape_r_SC, shape_r_SS, shape_r_SQ, shape_r_SD, shape__SO,
                     hands_HU, hands_HB,
                     direction_DS, direction_DD,
                     symmetry_S, symmetry_A]

def get_hands(df, row_num, num_hands_uni, num_hands_bi):
    if pd.notna(df.iloc[row_num, 11]):
        num_hands_uni += 1
    else:
        num_hands_bi += 1
    return (num_hands_uni, num_hands_bi)

def get_direction(df, row_num, num_direction_same, num_direction_diff):
    if pd.notna(df.iloc[row_num, 13]):
        num_direction_same += 1
    else:
        num_direction_diff += 1
    return (num_direction_same, num_direction_diff)

def get_symmetry(df, row_num, num_symm_s, num_symm_a):
    if pd.notna(df.iloc[row_num, 15]):
        num_symm_s += 1
    else:
        num_symm_a += 1
    return (num_symm_s, num_symm_a)

def get_shape(df, row_num, hand, num_shape_sc, num_shape_ss, num_shape_sq, num_shape_sd, num_shape_so):
    index_add = 0
    if hand == 'r':
        index_add = index_add + 5
    
    if pd.notna(df.iloc[row_num, 1+index_add]):
        num_shape_sc += 1
    elif pd.notna(df.iloc[row_num, 2+index_add]):
        num_shape_ss += 1
    elif pd.notna(df.iloc[row_num, 3+index_add]):
        num_shape_sq += 1
    elif pd.notna(df.iloc[row_num, 4+index_add]):
        num_shape_sd += 1
    elif pd.notna(df.iloc[row_num, 5+index_add]):
        num_shape_so += 1

    return (num_shape_sc, num_shape_ss, num_shape_sq, num_shape_sd, num_shape_so)

In [17]:
summary_df = pd.DataFrame()
initialize_dict(summary_df)

codebook_path = 'C:\\Users\\katie\\Documents\\cpsc\\VRelax\\gestureInterface\\Codebook'
for file in os.listdir(codebook_path):
    # Read individual codebook gesture files
    cb_file = os.path.join(codebook_path, file)
    print(cb_file)
    gesture_df = pd.read_csv(cb_file, header=None)
    row_length = gesture_df.shape[0]

    num_total_strokes = 0

    # Initializing all variables for df input
    gesture = get_gesture_name(file)
    num_hands_uni = 0
    num_hands_bi = 0
    num_direction_same = 0
    num_direction_diff = 0
    num_symm_s = 0
    num_symm_a = 0
    num_shape_l_sc = 0
    num_shape_l_ss = 0
    num_shape_l_sq = 0
    num_shape_l_sd = 0
    num_shape_l_so = 0
    num_shape_r_sc = 0
    num_shape_r_ss = 0
    num_shape_r_sq = 0
    num_shape_r_sd = 0
    num_shape_r_so = 0

    for i in range(row_length):
        # Only analyzes trial rows and Removes first three rows that contain the subject number, categories, and ids
        if (i % 8 != 0) and (i % 8 != 1) and (i % 8 != 2):
            num_total_strokes += 1
            num_hands_uni, num_hands_bi = get_hands(gesture_df, i, num_hands_uni, num_hands_bi)
            num_direction_same, num_direction_diff = get_direction(gesture_df, i, num_direction_same, num_direction_diff)
            num_symm_s, num_symm_a = get_symmetry(gesture_df, i, num_symm_s, num_symm_a)
            num_shape_l_sc, num_shape_l_ss, num_shape_l_sq, num_shape_l_sd, num_shape_l_so = get_shape(gesture_df, i, 'l', num_shape_l_sc, num_shape_l_ss, num_shape_l_sq, num_shape_l_sd, num_shape_l_so)
            num_shape_r_sc, num_shape_r_ss, num_shape_r_sq, num_shape_r_sd, num_shape_r_so = get_shape(gesture_df, i, 'r', num_shape_r_sc, num_shape_r_ss, num_shape_r_sq, num_shape_r_sd, num_shape_r_so)
    summary_df = summary_df.append({'gesture': gesture,
                       'shape l (SC)': get_percentage(num_shape_l_sc, num_total_strokes),
                       'shape l (SS)': get_percentage(num_shape_l_ss, num_total_strokes),
                       'shape l (SQ)': get_percentage(num_shape_l_sq, num_total_strokes), 
                       'shape l (SD)': get_percentage(num_shape_l_sd, num_total_strokes),
                       'shape l (SO)': get_percentage(num_shape_l_so, num_total_strokes), 
                       'shape r (SC)': get_percentage(num_shape_r_sc, num_total_strokes),
                       'shape r (SS)': get_percentage(num_shape_r_ss, num_total_strokes),
                       'shape r (SQ)': get_percentage(num_shape_r_sq, num_total_strokes), 
                       'shape r (SD)': get_percentage(num_shape_r_sd, num_total_strokes),
                       'shape r (SO)': get_percentage(num_shape_r_so, num_total_strokes), 
                       'hands (HU)': get_percentage(num_hands_uni, num_total_strokes),
                       'hands (HB)': get_percentage(num_hands_bi, num_total_strokes),
                       'direction (DS)': get_percentage(num_direction_same, num_total_strokes),
                       'direction (DD)': get_percentage(num_direction_diff, num_total_strokes),
                       'symmetry (S)': get_percentage(num_symm_s, num_total_strokes),
                       'symmetry (A)': get_percentage(num_symm_a, num_total_strokes),
    }, ignore_index = True)

display(summary_df)

export_csv(summary_df, codebook_path)
    
    


C:\Users\katie\Documents\cpsc\VRelax\gestureInterface\Codebook\Codebook_Pan_Down.csv
l
quarter
r
quarter
l
quarter
r
quarter
l
quarter
r
quarter
l
quarter
r
quarter
l
quarter
r
quarter
l
Dot
r
quarter
l
Dot
r
quarter
l
Dot
r
quarter
l
Dot
r
quarter
l
Dot
r
quarter
l
Dot
r
quarter
l
Dot
r
quarter
l
Dot
r
quarter
l
Dot
r
quarter
l
Dot
r
quarter
l
semicricle
r
semicricle
l
semicricle
r
semicricle
l
semicricle
r
semicricle
l
semicricle
r
semicricle
l
semicricle
r
semicricle
l
Dot
r
quarter
l
Dot
r
quarter
l
Dot
r
quarter
l
Dot
r
quarter
l
Dot
r
quarter
l
semicricle
r
semicricle
l
semicricle
r
semicricle
l
semicricle
r
semicricle
l
semicricle
r
semicricle
l
semicricle
r
semicricle
l
quarter
r
quarter
l
quarter
r
quarter
l
quarter
r
quarter
l
quarter
r
quarter
l
quarter
r
quarter
l
Dot
r
quarter
l
Dot
r
quarter
l
Dot
r
quarter
l
Dot
r
quarter
l
Dot
r
quarter
l
quarter
r
quarter
l
quarter
r
quarter
l
quarter
r
quarter
l
quarter
r
quarter
l
quarter
r
quarter
l
quarter
r
quarter
l
quarter
r
qua

,gesture,shape l (SC),shape l (SS),shape l (SQ),shape l (SD),shape l (SO),shape r (SC),shape r (SS),shape r (SQ),shape r (SD),shape r (SO),hands (HU),hands (HB),direction (DS),direction (DD),symmetry (S),symmetry (A)
0,PanDown,0.0%,14.4%,25.6%,60.0%,0.0%,0.0%,24.0%,72.0%,4.0%,0.0%,64.0%,36.0%,28.0%,72.0%,24.0%,76.0%
1,PanLeft,0.0%,7.2%,54.4%,38.4%,0.0%,0.0%,9.6%,83.2%,7.2%,0.0%,45.6%,54.4%,20.0%,80.0%,12.0%,88.0%
2,PanRight,0.0%,4.0%,48.0%,47.2%,0.8%,0.8%,12.0%,84.0%,2.4%,0.8%,48.8%,51.2%,26.4%,73.6%,10.4%,89.6%
3,PanUp,0.0%,8.0%,28.0%,64.0%,0.0%,0.0%,19.2%,76.0%,4.8%,0.0%,63.2%,36.8%,28.8%,71.2%,25.6%,74.4%
4,RotBackX,4.8%,50.4%,4.8%,40.0%,0.0%,21.6%,65.6%,12.8%,0.0%,0.0%,40.0%,60.0%,44.0%,56.0%,34.4%,65.6%
5,RotClockwiseY,4.8%,60.0%,10.4%,20.8%,4.0%,20.0%,72.0%,6.4%,0.0%,1.6%,20.8%,79.2%,12.8%,87.2%,0.0%,100.0%
6,RotCounterY,4.8%,37.6%,32.8%,21.6%,0.0%,22.4%,67.2%,8.8%,1.6%,0.0%,27.2%,72.8%,8.0%,92.0%,0.0%,100.0%
7,RotForwardX,4.0%,38.4%,16.8%,40.8%,0.0%,26.4%,53.6%,19.2%,0.0%,0.8%,39.2%,60.8%,32.8%,67.2%,32.0%,68.0%
8,RotLeftZ,0.0%,39.2%,35.2%,11.2%,14.4%,0.0%,36.0%,28.0%,36.0%,0.0%,20.0%,80.0%,14.4%,85.6%,0.0%,100.0%
9,RotRightZ,0.8%,20.0%,33.6%,45.6%,0.0%,12.0%,42.4%,45.6%,0.0%,0.0%,37.6%,62.4%,4.0%,96.0%,0.0%,100.0%
